# **Modeling**

In [50]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from scipy.stats import uniform

In [3]:
# Loading train and test variables into notebook
%store -r

In [4]:
# Verifying train and test variables are present
def func(var):
    if var in globals():
        return True

print(func('X_train'))
print(func('X_test'))
print(func('y_train'))
print(func('y_test'))

True
True
True
True


In [5]:
# Creating an out of the box model
model = LogisticRegression(random_state=10)
model.fit(X_train,y_train)

y_pred = model.predict(X_test)
model_accuracy  = accuracy_score(y_test, y_pred)

print("The Accuracy of simple logistic regression without hyperparmeters:",model_accuracy)
print("\n")
print(classification_report(y_test, y_pred))

The Accuracy of simple logistic regression without hyperparmeters: 0.9383561643835616


              precision    recall  f1-score   support

           0       0.94      1.00      0.97       960
           1       0.00      0.00      0.00        62

    accuracy                           0.94      1022
   macro avg       0.47      0.50      0.48      1022
weighted avg       0.88      0.94      0.91      1022



In [6]:
# Accuracy is high, but does not seem to be a good model
# Accuracy is high becuase there is a known class imbalace
# The model predicted a stroke event not occuring in all the data except one instance
print(np.unique(y_pred, return_counts=True))

(array([0, 1], dtype=int64), array([1021,    1], dtype=int64))


In [7]:
# Creating a balanced class model
log_model = LogisticRegression(class_weight='balanced', random_state=10)
log_model.fit(X_train,y_train)

y_pred = log_model.predict(X_test)
model_accuracy  = accuracy_score(y_test, y_pred)

print("The Accuracy of simple logistic regression with class weight parameter:",model_accuracy)
print("\n")
print(classification_report(y_test, y_pred))

The Accuracy of simple logistic regression with class weight parameter: 0.7583170254403131


              precision    recall  f1-score   support

           0       0.98      0.76      0.85       960
           1       0.17      0.76      0.28        62

    accuracy                           0.76      1022
   macro avg       0.57      0.76      0.57      1022
weighted avg       0.93      0.76      0.82      1022



In [8]:
# Accuracy decreased significantly, but improvements are seen in the precision and recall metrics
# Let's do some more hyperparameter tuning
# Some penalties are not compatiable with some solvers so three different models will be created

In [9]:
# Model 1
param_grid = [
    {'penalty':['l2',None],
    'C' : np.logspace(-4,4,20),
    'solver': ['lbfgs','newton-cg','sag'],
    'max_iter': [100,1000,2500,5000]
}
]
clf1 = GridSearchCV(log_model,param_grid = param_grid, cv = 3, verbose=True,n_jobs=-1, error_score='raise')

In [10]:
# Model 2
param_grid = [
    {'penalty':['l1','l2'],
    'C' : np.logspace(-4,4,20),
    'solver': ['liblinear'],
    'max_iter': [100,1000,2500,5000]
}
]
clf2 = GridSearchCV(log_model,param_grid = param_grid, cv = 3, verbose=True,n_jobs=-1, error_score='raise')

In [11]:
# Model 3
param_grid = [
    {'penalty':['l1','l2','elasticnet',None],
    'C' : np.logspace(-4,4,20),
    'solver': ['saga'],
    'max_iter': [100,1000,2500,5000],
    'l1_ratio': [0.10, 0.25, 0.5, 0.75, 0.9]
}
]
clf3 = GridSearchCV(log_model,param_grid = param_grid, cv = 3, verbose=True,n_jobs=-1, error_score='raise')

In [12]:
clf1.fit(X_train,y_train)
clf2.fit(X_train,y_train)
clf3.fit(X_train,y_train)

Fitting 3 folds for each of 480 candidates, totalling 1440 fits
Fitting 3 folds for each of 160 candidates, totalling 480 fits
Fitting 3 folds for each of 1600 candidates, totalling 4800 fits


GridSearchCV(cv=3, error_score='raise',
             estimator=LogisticRegression(class_weight='balanced',
                                          random_state=10),
             n_jobs=-1,
             param_grid=[{'C': array([1.00000000e-04, 2.63665090e-04, 6.95192796e-04, 1.83298071e-03,
       4.83293024e-03, 1.27427499e-02, 3.35981829e-02, 8.85866790e-02,
       2.33572147e-01, 6.15848211e-01, 1.62377674e+00, 4.28133240e+00,
       1.12883789e+01, 2.97635144e+01, 7.84759970e+01, 2.06913808e+02,
       5.45559478e+02, 1.43844989e+03, 3.79269019e+03, 1.00000000e+04]),
                          'l1_ratio': [0.1, 0.25, 0.5, 0.75, 0.9],
                          'max_iter': [100, 1000, 2500, 5000],
                          'penalty': ['l1', 'l2', 'elasticnet', None],
                          'solver': ['saga']}],
             verbose=True)

In [13]:
print(clf1.best_params_) 
print(clf2.best_params_)
print(clf3.best_params_)

{'C': 0.0001, 'max_iter': 100, 'penalty': 'l2', 'solver': 'newton-cg'}
{'C': 0.0001, 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'}
{'C': 0.0001, 'l1_ratio': 0.1, 'max_iter': 100, 'penalty': None, 'solver': 'saga'}


In [14]:
print(f'Accuracy of model 1- : {clf1.score(X_test,y_test):.5f}')
print(f'Accuracy of model 2- : {clf2.score(X_test,y_test):.5f}')
print(f'Accuracy of model 3- : {clf3.score(X_test,y_test):.5f}')

Accuracy of model 1- : 0.74168
Accuracy of model 2- : 0.93933
Accuracy of model 3- : 0.75832


In [15]:
y_pred1 = clf1.predict(X_test)
y_pred2 = clf2.predict(X_test)
y_pred3 = clf3.predict(X_test)

print("*************************Model 1*************************")
print(classification_report(y_test, y_pred1))

print("*************************Model 2*************************")
print(classification_report(y_test, y_pred2))

print("*************************Model 3*************************")
print(classification_report(y_test, y_pred3))

*************************Model 1*************************
              precision    recall  f1-score   support

           0       0.98      0.74      0.84       960
           1       0.16      0.77      0.27        62

    accuracy                           0.74      1022
   macro avg       0.57      0.76      0.55      1022
weighted avg       0.93      0.74      0.81      1022

*************************Model 2*************************
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       960
           1       0.00      0.00      0.00        62

    accuracy                           0.94      1022
   macro avg       0.47      0.50      0.48      1022
weighted avg       0.88      0.94      0.91      1022

*************************Model 3*************************
              precision    recall  f1-score   support

           0       0.98      0.76      0.85       960
           1       0.17      0.76      0.28        62

    accu

In [48]:
# Using Random Grid Search

In [61]:
# Model 1
param_grid = [
    {'penalty':['l2',None],
    'C' : uniform(0, 4),
    'solver': ['lbfgs','newton-cg','sag'],
    'max_iter': [100,1000,2500,5000]
}
]
clf1 = RandomizedSearchCV(log_model,param_distributions = param_grid, cv = 5, verbose=True,n_jobs=-1, error_score='raise')

In [62]:
# Model 2
param_grid = [
    {'penalty':['l1','l2'],
    'C' : uniform(0, 4),
    'solver': ['liblinear'],
    'max_iter': [100,1000,2500,5000]
}
]
clf2 = RandomizedSearchCV(log_model,param_distributions = param_grid, cv = 5, verbose=True,n_jobs=-1, error_score='raise')

In [63]:
# Model 3
param_grid = [
    {'penalty':['l1','l2','elasticnet',None],
    'C' : uniform(0, 4),
    'solver': ['saga'],
    'max_iter': [100,1000,2500,5000],
    'l1_ratio': uniform(0, 1)
}
]
clf3 = RandomizedSearchCV(log_model,param_distributions = param_grid, cv = 5, verbose=True,n_jobs=-1, error_score='raise')

In [64]:
clf1.fit(X_train,y_train)
clf2.fit(X_train,y_train)
clf3.fit(X_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5, error_score='raise',
                   estimator=LogisticRegression(class_weight='balanced',
                                                random_state=10),
                   n_jobs=-1,
                   param_distributions=[{'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000020FE5121990>,
                                         'l1_ratio': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000020FE51227D0>,
                                         'max_iter': [100, 1000, 2500, 5000],
                                         'penalty': ['l1', 'l2', 'elasticnet',
                                                     None],
                                         'solver': ['saga']}],
                   verbose=True)

In [65]:
print(clf1.best_params_) 
print(clf2.best_params_)
print(clf3.best_params_)

{'C': 0.8826421019029382, 'max_iter': 2500, 'penalty': 'l2', 'solver': 'newton-cg'}
{'C': 2.728967135599356, 'max_iter': 2500, 'penalty': 'l2', 'solver': 'liblinear'}
{'C': 2.2473370510323027, 'l1_ratio': 0.8362046907394218, 'max_iter': 1000, 'penalty': 'elasticnet', 'solver': 'saga'}


In [66]:
print(f'Accuracy of model 1- : {clf1.score(X_test,y_test):.5f}')
print(f'Accuracy of model 2- : {clf2.score(X_test,y_test):.5f}')
print(f'Accuracy of model 3- : {clf3.score(X_test,y_test):.5f}')

Accuracy of model 1- : 0.75832
Accuracy of model 2- : 0.75832
Accuracy of model 3- : 0.75832


In [67]:
y_pred1 = clf1.predict(X_test)
y_pred2 = clf2.predict(X_test)
y_pred3 = clf3.predict(X_test)

print("*************************Model 1*************************")
print(classification_report(y_test, y_pred1))

print("*************************Model 2*************************")
print(classification_report(y_test, y_pred2))

print("*************************Model 3*************************")
print(classification_report(y_test, y_pred3))

*************************Model 1*************************
              precision    recall  f1-score   support

           0       0.98      0.76      0.85       960
           1       0.17      0.76      0.28        62

    accuracy                           0.76      1022
   macro avg       0.57      0.76      0.57      1022
weighted avg       0.93      0.76      0.82      1022

*************************Model 2*************************
              precision    recall  f1-score   support

           0       0.98      0.76      0.85       960
           1       0.17      0.76      0.28        62

    accuracy                           0.76      1022
   macro avg       0.57      0.76      0.57      1022
weighted avg       0.93      0.76      0.82      1022

*************************Model 3*************************
              precision    recall  f1-score   support

           0       0.98      0.76      0.85       960
           1       0.17      0.76      0.28        62

    accu

In [68]:
# Remove all variables from storage
# %store -z